# Sparse Autoencoders

# Sparse Autoencoders

## What This Is
Sparse Autoencoders (SAEs) are used to find *monosemantic* features in neural networks. The idea: if polysemantic neurons represent multiple features in superposition, train a sparse autoencoder to *decompose* the neuron's activations into a larger dictionary of sparse, monosemantic features.

**Architecture**: f(x) = ReLU(W_enc(x - b_dec) + b_enc); x̂ = W_dec f(x) + b_dec

**Sparsity constraint**: L1 penalty on the hidden layer forces few features to be active at once, discovering the sparse underlying features.

**Key insight**: If the original network uses superposition to pack 100 features into 10 neurons, the SAE can recover those 100 features in 100 monosemantic SAE features.

In [ ]:
import numpy as np

np.random.seed(42)

class SparseAutoencoder:
    def __init__(self, input_dim: int, n_features: int, lr: float = 0.001, l1_coef: float = 0.01):
        self.input_dim = input_dim
        self.n_features = n_features
        self.lr = lr
        self.l1_coef = l1_coef
        # Encoder
        self.W_enc = np.random.randn(input_dim, n_features) * 0.1
        self.b_enc = np.zeros(n_features)
        # Decoder (constrained to unit norm columns)
        self.W_dec = np.random.randn(n_features, input_dim) * 0.1
        self._normalize_decoder()
        self.b_dec = np.zeros(input_dim)
    
    def _normalize_decoder(self):
        norms = np.linalg.norm(self.W_dec, axis=1, keepdims=True)
        self.W_dec = self.W_dec / (norms + 1e-8)
    
    def encode(self, x):
        return np.maximum(0, (x - self.b_dec) @ self.W_enc + self.b_enc)
    
    def decode(self, f):
        return f @ self.W_dec + self.b_dec
    
    def forward(self, x):
        f = self.encode(x)
        return self.decode(f), f
    
    def train_step(self, X_batch):
        n = len(X_batch)
        # Forward
        pre_enc = (X_batch - self.b_dec) @ self.W_enc + self.b_enc
        f = np.maximum(0, pre_enc)
        x_hat = f @ self.W_dec + self.b_dec
        
        # Losses
        recon_loss = ((x_hat - X_batch)**2).mean()
        l1_loss = np.abs(f).mean()
        total = recon_loss + self.l1_coef * l1_loss
        
        # Gradients
        d_recon = 2 * (x_hat - X_batch) / n
        d_dec = f.T @ d_recon / n + self.l1_coef * np.sign(f).T @ np.ones((n, self.input_dim)) / n
        d_b_dec = d_recon.mean(0)
        d_f = d_recon @ self.W_dec.T
        d_f_pre = d_f * (pre_enc > 0)
        d_enc = (X_batch - self.b_dec).T @ d_f_pre / n
        d_b_enc = d_f_pre.mean(0)
        d_b_dec += -(self.W_enc @ d_f_pre.T).mean(1)  # gradient through b_dec
        
        self.W_enc -= self.lr * d_enc
        self.b_enc -= self.lr * d_b_enc
        self.W_dec -= self.lr * d_dec
        self.b_dec -= self.lr * d_b_dec * 0  # skip b_dec for simplicity
        self._normalize_decoder()
        
        return recon_loss, l1_loss

# Generate synthetic activations from a polysemantic MLP
# 4 neurons, 8 underlying features
true_features = np.random.randn(8, 4) * 0.5  # 8 features in 4D space
feature_weights = np.random.randn(4)  # decoder

N = 5000
# Sparse feature activations
F_true = np.random.exponential(1, (N, 8)) * (np.random.random((N, 8)) > 0.8)
# Neural activations = mix of features
X_acts = np.maximum(0, F_true @ true_features + np.random.randn(N, 4) * 0.1)

# Train SAE to recover the 8 features
sae = SparseAutoencoder(input_dim=4, n_features=8, l1_coef=0.05)

losses = []
for step in range(1000):
    idx = np.random.randint(0, N, 128)
    rl, l1 = sae.train_step(X_acts[idx])
    if step % 200 == 0:
        losses.append((step, rl, l1))

print('Sparse Autoencoder Training:')
for step, rl, l1 in losses:
    print(f'  Step {step:4d}: recon_loss={rl:.4f}, l1={l1:.4f}')

# Evaluate: how sparse are the learned features?
f_test, _ = sae.forward(X_acts[:500])
# Wait, forward returns (decoded, features), let me fix
_, f_test = sae.forward(X_acts[:500])
sparsity = (f_test < 0.01).mean()
print(f'\nFeature sparsity: {sparsity:.3f} ({sparsity*100:.1f}% of feature-activations are zero)')
print(f'Average active features per sample: {(f_test > 0.01).sum(1).mean():.1f}')
